<a href="https://colab.research.google.com/github/cliteka-cell/ner-wikiann/blob/main/exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets evaluate accelerate seqeval

from datasets import load_dataset

dataset = load_dataset('wikiann', 'en')
print(dataset)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.2 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

en/validation-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/test-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})


In [ ]:
# Get label names
label_names = dataset['train'].features['ner_tags'].feature.names
print('Labels:', label_names)

Labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


In [ ]:
# Look at one example — like an annotation view
example = dataset['train'][0]

for token, tag in zip(example['tokens'], example['ner_tags']):
    print(f"{token:20} {label_names[tag]}")

R.H.                 B-ORG
Saunders             I-ORG
(                    O
St.                  B-ORG
Lawrence             I-ORG
River                I-ORG
)                    O
(                    O
968                  O
MW                   O
)                    O


In [ ]:
from collections import Counter

all_tags = [tag for row in dataset['train']['ner_tags'] for tag in row]
counts = Counter(all_tags)

print(f"{'Label':12} {'Count':>8}  {'%':>6}")
print('-' * 30)
for tag_id, count in sorted(counts.items()):
    print(f"{label_names[tag_id]:12} {count:>8}  {count/len(all_tags)*100:>5.1f}%")

Label           Count       %
------------------------------
O               81362   50.7%
B-PER            9164    5.7%
I-PER           14698    9.2%
B-ORG            9422    5.9%
I-ORG           23226   14.5%
B-LOC            9345    5.8%
I-LOC           13177    8.2%


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )
    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        aligned = []
        prev = None
        for word_id in word_ids:
            if word_id is None:
                aligned.append(-100)      # special tokens → ignored in loss
            elif word_id != prev:
                aligned.append(labels[word_id])
            else:
                aligned.append(-100)      # subword continuation → ignored
            prev = word_id
        all_labels.append(aligned)
    tokenized["labels"] = all_labels
    return tokenized

tokenized_dataset = dataset.map(tokenize_and_align, batched=True)
print("Done!", tokenized_dataset)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Done! DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 20000
    })
})


In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_names),
    id2label={i: l for i, l in enumerate(label_names)},
    label2id={l: i for i, l in enumerate(label_names)}
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    true_labels = [
        [label_names[l] for l in label if l != -100]
        for label in labels
    ]
    true_preds = [
        [label_names[p] for p, l in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]
    results = seqeval.compute(predictions=true_preds, references=true_labels)
    return {"f1": results["overall_f1"], "accuracy": results["overall_accuracy"]}

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

args = TrainingArguments(
    output_dir="ner-results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.302787,0.260332,0.797976,0.922035
2,0.213362,0.246659,0.815121,0.928032
3,0.159607,0.252214,0.820165,0.929100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3750, training_loss=0.24929791158040365, metrics={'train_runtime': 331.6582, 'train_samples_per_second': 180.909, 'train_steps_per_second': 11.307, 'total_flos': 462214875911712.0, 'train_loss': 0.24929791158040365, 'epoch': 3.0})

In [ ]:
results = trainer.evaluate()
print(f"F1 Score:  {results['eval_f1']:.4f}")
print(f"Accuracy:  {results['eval_accuracy']:.4f}")
print(f"Loss:      {results['eval_loss']:.4f}")

F1 Score:  0.8202
Accuracy:  0.9291
Loss:      0.2522


In [ ]:
from transformers import pipeline

ner = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

sentence = "Abraham Lincoln was born in Larue County, KY and Lincoln Memorial in Washington D.C. was built for him by Henry Bacon."
results_ner = ner(sentence)

for entity in results_ner:
    print(f"{entity['word']:20} {entity['entity_group']:8} {entity['score']:.2f}")

abraham lincoln      PER      0.76
larue county         LOC      0.95
ky                   LOC      0.50
lincoln memorial     ORG      0.94
washington d. c.     LOC      0.75
henry bacon          PER      0.91
